# Rossmann Retail Forecasting & Product Analytics

This notebook is the review layer for the modular pipeline. It frames the forecasting task as a Product Data Science problem: define forecast and guardrail metrics, use a time-based holdout, diagnose segment performance, investigate promotions and unusual sales changes, and translate outputs into decisions.

**Important:** promotion lift in this project is observational. It is a hypothesis-generating metric, not a causal estimate.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from rossmann_analytics import ProjectConfig, run_pipeline

config = ProjectConfig(project_root=PROJECT_ROOT, validation_days=42, run_shap=True)
config

## Run the reproducible pipeline

The pipeline validates the raw files, creates EDA figures, builds leakage-safe features, trains three model tiers, performs segment diagnostics, computes descriptive promotion lift, flags sales spikes and drops, and writes an automated report.

In [ ]:
result = run_pipeline(config)
result

## Data quality and exploratory analysis

In [ ]:
quality = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'data_quality_summary.csv')
quality.head(15)

In [ ]:
for filename in [
    'sales_distribution.png',
    'monthly_sales_trend.png',
    'sales_by_promotion.png',
    'sales_by_day_of_week.png',
    'competition_distance_vs_sales.png',
]:
    display(Image(filename=str(PROJECT_ROOT / 'reports' / 'figures' / filename), width=850))

## Time-based model comparison

RMSPE is the primary comparison metric because it reflects relative forecast error and matches the original competition. MAE and RMSE remain visible so a low-volume store does not dominate the entire interpretation.

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'model_comparison.csv')
comparison.style.format({'mae': '{:,.1f}', 'rmse': '{:,.1f}', 'rmspe': '{:.4f}'})

## Segment guardrails

A model can look strong in aggregate while failing during promotions, holidays, or for specific store formats. The following table ranks cohorts by RMSPE for targeted investigation.

In [ ]:
segments = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'segment_performance.csv')
segments.sort_values('rmspe', ascending=False).head(20)

## Promotion analysis and experimentation framing

The descriptive lift compares promotion sales with each store's historical non-promotion mean. Because promotion assignment is not random, the result should guide experiment design rather than be presented as incremental causal impact. A follow-up store-level test would use incremental sales or gross profit as the primary metric and margin, inventory, stockouts, cannibalization, and post-promotion demand as guardrails.

In [ ]:
promotion = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'promotion_lift.csv')
promotion.style.format({
    'mean_observational_lift': '{:.1%}',
    'median_observational_lift': '{:.1%}',
    'mean_promo_sales': '{:,.0f}',
    'mean_store_nonpromo_baseline': '{:,.0f}',
})

## Root-cause review

Extreme store-days are detected relative to a trailing 28-day baseline. The retained context supports investigation, but the pipeline deliberately says a factor is *associated with* an event rather than assigning cause automatically.

In [ ]:
events = pd.read_csv(PROJECT_ROOT / 'reports' / 'tables' / 'sales_spike_drop_events.csv')
events.head(20)

## Evidence-grounded report

In [ ]:
display(Markdown((PROJECT_ROOT / 'reports' / 'model_report.md').read_text(encoding='utf-8')))